In [1]:
import pandas as pd
import glob

files = glob.glob("extracted_data/group*/experiment*/subject*.csv")
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(df.shape)
df.head()
# df.to_parquet("dipser_data.parquet", engine="pyarrow", index=False)

(500345, 30)


,group,time,subject,experiment,image_path,metadata,labeler_02 emotion,labeler_02 attention,labeler_01 attention,labeler_04 attention,...,labeler_04 emotionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,self_labeling emotionfilled,self_labeling attentionfilled,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled
0,group01,10:59:49.000776,subject_01,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,9.0,3.0,2.0,3.0,...,8.0,8,3,9,9,3,NaN,NaN,NaN,NaN
1,group01,10:59:49.147013,subject_01,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,8.0,8,3,9,9,3,NaN,NaN,NaN,NaN
2,group01,10:59:49.252899,subject_01,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,8.0,8,3,9,9,3,NaN,NaN,NaN,NaN
3,group01,10:59:49.343618,subject_01,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,8.0,8,3,9,9,3,NaN,NaN,NaN,NaN
4,group01,10:59:49.504931,subject_01,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,8.0,8,3,9,9,3,NaN,NaN,NaN,NaN


The following cell was executed in Snellius cluster where all the metadata folders for each subject are stored. As an output we receive the same df + 3 columns of age, gender and race for each row. It will output the file "dipser_data_metadata.parquet" which we use in the subsequent cells.

In [ ]:
# Metadata extraction function
def extract_metadata(json_path):
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        face = data.get("person", {}).get("face", {})

        age = face.get("age", None)
        gender = face.get("gender", {}).get("gender_name", None)
        race = face.get("race", {}).get("dominant_race", None)

        return age, gender, race

    except Exception:
        return None, None, None

# Parallel processing
def process_dataframe(df, n_jobs):
    paths = df["metadata"].tolist()
    results = []

    for i, p in enumerate(paths):
        results.append(extract_metadata(p))

        if i % 5000 == 0:
            print(f"Processed {i}/{len(paths)} rows")

    df[["age", "gender_name", "race"]] = pd.DataFrame(results, index=df.index)
    return df

def main():
    start_time = time.time()
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="Input dataframe path (parquet)")
    parser.add_argument("--output", required=True, help="Output dataframe path")
    parser.add_argument("--n_jobs", type=int, default=16)

    args = parser.parse_args()

    print("Loading dataframe...")
    df = pd.read_parquet(args.input)

    print(f"Processing {len(df)} rows with {args.n_jobs} workers...")

    df = process_dataframe(df, args.n_jobs)

    print("Saving output...")
    df.to_parquet(args.output)

    print("Done.")
    print(f"Elapsed: {time.time() - start_time:.2f}s")

In [2]:
df = pd.read_parquet("dipser_data_metadata.parquet", engine="pyarrow")

In [3]:
"""Deepface extracts per each timeframe the estimation for race, gender and age.
across rows the label is not consistent (e.g. in one timeframe the person might be classified as White, and in other timeframe 
as Middle Eastern. For this reason, we will keep the most common label for the whole subject."""

# we will groupby group experiment and subject and we will keep the most dominant 
def get_mode(series):
    return series.dropna().mode().iloc[0] if not series.dropna().empty else None

In [4]:
subject_metadata  = (
    df.groupby(["group", "experiment", "subject"])
    .agg({
        "race": get_mode,
        "gender_name": get_mode,
        "age": "mean"
    })
    .reset_index()
)

In [5]:
df = df.drop(columns=['race', 'gender_name', 'age']) # will be replaced with the new values
df = df.merge(
    subject_metadata,
    on=["group", "experiment", "subject"],
    how="left"
)

In [7]:
df['age'] = round(df['age'])

In [8]:
df.to_parquet("dipser_transformed_data.parquet", engine="pyarrow", index=False)

Next step: Add sensor data